# Diff-SSL TVC-LSTM Baseline — Multi-Setting Direct Output

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo and mounts
Drive for the dataset. **Push local changes before running.**

Ablation **base model**: the conditioned LSTM from the diffssl paper recipe
(`LSTM32TVC` / `LSTM96TVC` — `cond_type="tvcond"`, `TVFiLMCond` + sample-rate
LSTM, `0.5·L1 + 0.5·MR-STFT`, AdamW + ReduceLROnPlateau, TBPTT
`step_num_samples=4410`), built via the published [nablafx](https://github.com/mcomunita/nablafx)
package (`pip install nablafx`; the gitignored `external/` checkout is not in
this repo).

**Dataset / split** — **v2 external `test_ground_truth` policy**, shared with the
`06_output` notebooks (GR-TFiLM, coloration black-box) so every model in the
ablation scores on the **same held-out test set**:
- 10 settings × 10 songs (GR-curve inventory gates pairs; wet WAV is the target)
- seed 42: the held-out test set is exactly the `test_ground_truth/` pairs,
  excluded from train/val **by key**; val = 1 remaining song × all settings;
  every other (song, setting) pair — including the former lowest-threshold test
  songs/settings — is training material.

**Training recipe** — matches diffssl `LSTM32TVC` / `BlackBoxSystemWithTBPTT`:
- 3 s crops (`sample_length=132300`), `batch_size=64`, train shuffle + `drop_last`
- LSTM state **reset every batch**; TBPTT sub-steps of `4410` samples inside each crop
- `0.5·L1 + 0.5·MR-STFT`, AdamW + ReduceLROnPlateau

**Training budget**: fixed **100 epochs**.

### L4-speed tuning (deviates from the diffssl `batch_size=16` recipe)

Same tuning as [`08_la2a/train_lstm_la2a_tvc.ipynb`](../08_la2a/train_lstm_la2a_tvc.ipynb).
This LSTM is tiny (~8k params); its cost is the sample-rate recurrence, whose
per-step GPU latency is ~flat across batch sizes, so an L4 is badly
underutilised at batch 16 and epoch wall-clock scales roughly `1/batch_size`.
We therefore run **`batch_size=64`** (≈4× fewer, bigger batches → far better L4
utilisation) with the LR **sqrt-scaled to `2e-3`** (`1e-3·√(64/16)`) to keep the
per-epoch update count healthy at the fixed 100-epoch budget. Set
`BATCH_SIZE = 16`, `LR = 1e-3` below to fall back to the exact diffssl recipe.

In [1]:
# ── 0. Dependencies ──────────────────────────────────────────────────
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

# diffssl nablafx.system imports FAD — stub so we never pull tensorflow/encodec.
fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} — restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 69.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 kB 29.9 MB/s eta 0:00:00
numpy 2.0.2, torch 2.11.0+cu128


In [ ]:
# ── 1. Mount Drive + clone repo ─────────────────────────────────────
# Model code comes from pip ``nablafx`` (cell 0). This repo only supplies
# dataset/system helpers under 02b_sota_training/ (external/ is gitignored).

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"
SOTA_DIR = os.path.join(REPO_ROOT, "02b_sota_training")
OUTPUT_DIR = os.path.join(os.path.dirname(DRIVE_DATA_ROOT), "diffssl_tvc_runs")

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" reset --hard origin/main
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT
_dataset_py = os.path.join(SOTA_DIR, "dataset.py")
assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "test_ground_truth")), (
    f"No test_ground_truth/ under {DATA_ROOT} — it defines the held-out test set "
    "(v2 split policy, shared with 06_output); sync it to Drive before training."
)
assert os.path.isfile(_dataset_py), f"Clone failed: {REPO_ROOT}"
_src = open(_dataset_py).read()
assert "BATCH_SIZE" in _src and "DiffSSLCropDataModule" in _src and "test_gt_root" in _src, (
    "dataset.py on disk is stale (needs the test_gt_root external-test support) — "
    f"re-run this cell; if it persists, delete {REPO_ROOT} and clone again"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop cached local modules so a prior run cannot keep the old stateful dataset.
for _name in list(sys.modules):
    if _name in ("dataset", "splits", "system", "model"):
        del sys.modules[_name]

for p in (REPO_ROOT, SOTA_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"SOTA_DIR   : {SOTA_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

In [ ]:
# ── 2. Cache dataset to Colab local SSD ──────────────────────────────
# Mirror dry WAV per song + wet WAV per (song, setting) pair to local SSD, with
# wet/gr copied under their ORIGINAL subfolders (processed_ground_truth vs
# test_ground_truth) so the external test pairs stay under test_ground_truth/
# on the cache too (the v2 split treats that folder as test-only). gr_curves
# are copied for split/inventory parity — the LSTM32TVC model never reads them.

import shutil
from dataset import discover_diffssl_wet_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_diffssl_wet_pairs(DATA_ROOT, include_test_ground_truth=True)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs × {len(settings)} settings ({len(pairs)} pairs) → {LOCAL_DATA_ROOT}")

def _mirror(src, dst):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# dry WAVs (one per song, shared across settings)
for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    _mirror(Path(DATA_ROOT) / "processed_normalized" / fn,
            Path(LOCAL_DATA_ROOT) / "processed_normalized" / fn)

# GR curves (.pt) + wet WAVs (-exported.wav), per pair, preserving the source
# subfolder (processed_ground_truth vs test_ground_truth)
for p in pairs:
    _mirror(p["gr"], Path(LOCAL_DATA_ROOT) / Path(p["gr"]).relative_to(DATA_ROOT))
    _mirror(p["wet"], Path(LOCAL_DATA_ROOT) / Path(p["wet"]).relative_to(DATA_ROOT))

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

In [ ]:
# ── 3. Imports & hyper-parameters (LSTM32TVC / LSTM96TVC) ──────────

import importlib
import json
from datetime import datetime

import lightning as pl
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

import dataset as _dataset
importlib.reload(_dataset)
from dataset import (
    SAMPLE_LENGTH,
    SAMPLE_RATE,
    DiffSSLCropDataModule,
    discover_diffssl_wet_pairs,
)  # BATCH_SIZE is set explicitly below (L4 tuning), not imported

import model as _model
importlib.reload(_model)
from model import build_diffssl_tvc_lstm

import system as _system
importlib.reload(_system)
from system import DiffSSLTVCLSTMSystem

from splits import (
    DIFFSSL_PARAM_RANGES, build_split_manifest, discover_test_ground_truth_keys,
)
from src.dsp import PARAM_ORDER

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split: v2 external test_ground_truth policy (shared with 06_output) — the
#    ONLY held-out test set is test_ground_truth/, excluded from train/val by
#    key; every other (song, setting) pair trains or validates. --
SPLIT_SEED = 42
N_VAL_SONGS = 1

# -- training recipe: diffssl LSTM32TVC / BlackBoxSystemWithTBPTT --
# L4-speed tuning (mirrors 08_la2a/train_lstm_la2a_tvc.ipynb): the diffssl recipe
# is batch 16 / LR 1e-3. This LSTM is tiny and the L4 is idle most of the
# sample-rate recurrence, so we run a bigger batch (epoch wall-clock ~ 1/batch)
# with the LR sqrt-scaled to keep 100-epoch convergence. Set BATCH_SIZE=16,
# LR=1e-3 to reproduce the exact diffssl recipe.
BATCH_SIZE = 64                  # diffssl uses 16; 64 far better utilises the L4
LR = 2e-3                        # sqrt-scaled: 1e-3 * sqrt(64/16)
MAX_EPOCHS = 100
STEP_NUM_SAMPLES = 4410          # diffssl LSTM TBPTT sub-step (0.1 s @ 44.1 kHz)

HIDDEN_SIZE = 32                 # LSTM32TVC; set 96 for LSTM96TVC
NUM_LAYERS = 1
COND_TYPE = "tvcond"
COND_BLOCK_SIZE = 128
COND_NUM_LAYERS = 1
NUM_CONTROLS = 4

RUN_TAG = "diffssl_lstm32_tvc_multisetting"
RESUME_RUN = None

In [ ]:
# ── 4. Preview split — v2 external test_ground_truth policy ──────────
# The ONLY held-out test set is test_ground_truth/ (scanned on DRIVE; the local
# cache mirrors it under the same subfolder). Its pairs are excluded from
# train/val BY KEY; every remaining (song, setting) pair trains or validates —
# including the former lowest-threshold test songs/settings.

TEST_KEYS = discover_test_ground_truth_keys(DRIVE_DATA_ROOT)
print(f"test_ground_truth pairs ({len(TEST_KEYS)}):")
for k in sorted(TEST_KEYS):
    print(f"  {k}")

preview = build_split_manifest(
    discover_diffssl_wet_pairs(DATA_ROOT, include_test_ground_truth=True),
    seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS,
    test_pair_keys=TEST_KEYS,
)
leaked = (set(preview.train_pair_keys) | set(preview.val_pair_keys)) & TEST_KEYS
assert not leaked, f"test_ground_truth pairs leaked into train/val: {sorted(leaked)}"

print(f"\nSettings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test pairs : {preview.test_pair_keys}")
print(
    f"Pairs — train={len(preview.train_pair_keys)} "
    f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}"
)

In [ ]:
# ── 5. Build diffssl model (LSTM32TVC / LSTM96TVC) ─────────────────

model = build_diffssl_tvc_lstm(
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_controls=NUM_CONTROLS,
    cond_block_size=COND_BLOCK_SIZE,
    cond_num_layers=COND_NUM_LAYERS,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"BlackBoxModel + LSTM(tvcond, h={HIDDEN_SIZE}): {n_params:,} params")
print(model.processor)


BlackBoxModel:
LSTM(
  (cond_nn): TVFiLMCond(
    (pool): MaxPool1d(kernel_size=128, stride=128, padding=0, dilation=1, ceil_mode=False)
    (lstm): LSTM(5, 16)
  )
  (lstm): LSTM(17, 32)
  (lin): Linear(in_features=32, out_features=1, bias=True)
)

BlackBoxModel + LSTM(tvcond, h=32): 8,033 params
LSTM(
  (cond_nn): TVFiLMCond(
    (pool): MaxPool1d(kernel_size=128, stride=128, padding=0, dilation=1, ceil_mode=False)
    (lstm): LSTM(5, 16)
  )
  (lstm): LSTM(17, 32)
  (lin): Linear(in_features=32, out_features=1, bias=True)
)


In [ ]:
# ── 6. Train ─────────────────────────────────────────────────────────

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

# Multi-worker data loading: the model is tiny (8k params), so without this the
# GPU starves on the per-item soundfile seeks (dry + wet). Cap at 8.
NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"DataLoader num_workers: {NUM_WORKERS}")

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"diffssl_tvc_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = DiffSSLCropDataModule(
    data_root=DATA_ROOT,
    sample_length=SAMPLE_LENGTH,
    sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE,
    split_seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS,
    test_gt_root=DRIVE_DATA_ROOT,   # test keys scanned on Drive (source of truth)
    split_manifest_path=split_path,
    num_workers=NUM_WORKERS,
)
dm.setup()

print(f"Train/val/test crops: {len(dm.train_dataset)} / {len(dm.val_dataset)} / {len(dm.test_dataset)}")
print(f"Batches/epoch (train): {len(dm.train_dataloader())}  (batch_size={BATCH_SIZE})")
print(f"Split manifest: {split_path}")

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "diffssl_direct_output_lstm_tvcond",
        "model_type": "nablafx_diffssl_LSTM_tvcond",
        "model_ref": f"experiments/LSTM{HIDDEN_SIZE}TVC/config.yaml",
        "dataset": "Diff-SSL-G-Comp",
        "setting": "multi (all non-test settings)",
        "conditioning": "tvcond (TVFiLMCond)",
        "sample_rate": SAMPLE_RATE,
        "sample_length": SAMPLE_LENGTH,
        "batch_size": BATCH_SIZE,
        "step_num_samples": STEP_NUM_SAMPLES,
        "param_order": PARAM_ORDER,
        "param_ranges": DIFFSSL_PARAM_RANGES,
        "split_seed": SPLIT_SEED,
        "split_policy": "external_test_ground_truth (test pairs excluded from train/val by key)",
        "test_pair_keys": sorted(TEST_KEYS),
        "test_settings": dm.split.test_settings,
        "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs,
        "test_songs": dm.split.test_songs,
        "model": {
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "cond_type": COND_TYPE,
            "cond_block_size": COND_BLOCK_SIZE,
            "cond_num_layers": COND_NUM_LAYERS,
            "num_controls": NUM_CONTROLS,
            "num_params": n_params,
        },
        "lr": LR,
        "max_epochs": MAX_EPOCHS,
        "loss": "0.5*L1 + 0.5*MR-STFT",
        "optimizer": "adamw + reducelronplateau(0.5,p20)",
        "training": "diffssl_crop_batches + tbptt_substeps (reset each batch)",
        "l4_speed_tuning": (
            f"batch {BATCH_SIZE} + LR {LR} (sqrt-scaled) vs diffssl batch 16 / LR 1e-3; "
            "epoch wall-clock ~ 1/batch for this sample-rate LSTM"
        ),
    }, f, indent=2)

system = DiffSSLTVCLSTMSystem(
    model=model,
    lr=LR,
    step_num_samples=STEP_NUM_SAMPLES,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(
        dirpath=ckpt_dir,
        monitor="loss/val",
        mode="min",
        save_top_k=3,
        save_last=True,
        filename="best-{epoch:03d}-{step}",
        auto_insert_metric_name=False,
    ),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu",
    devices=1,
    callbacks=callbacks,
    logger=loggers,
    log_every_n_steps=10,
)

trainer.fit(system, dm, ckpt_path=_resume_ckpt)

In [ ]:
# ── 7. Test (optional) ───────────────────────────────────────────────

best_ckpt = callbacks[0].best_model_path or os.path.join(ckpt_dir, "last.ckpt")
print(f"Testing with: {best_ckpt}")
trainer.test(system, dm, ckpt_path=best_ckpt)

INFO: Restoring states from the checkpoint path at /content/drive/Othercomputers/MacBook Air/data/diffssl_tvc_runs/diffssl_tvc_20260624_153343_diffssl_lstm32_tvc_multisetting/checkpoints/best-096-1181460.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/Othercomputers/MacBook Air/data/diffssl_tvc_runs/diffssl_tvc_20260624_153343_diffssl_lstm32_tvc_multisetting/checkpoints/best-096-1181460.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at /content/drive/Othercomputers/MacBook Air/data/diffssl_tvc_runs/diffssl_tvc_20260624_153343_diffssl_lstm32_tvc_multisetting/checkpoints/best-096-1181460.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Loaded model weights from the checkpoint at /content/drive/Othercomputers/MacBook Air/data/diffssl_tvc_runs/diffssl_tvc_20260624_153343_diffssl_lstm32_tvc_multiset

Testing with: /content/drive/Othercomputers/MacBook Air/data/diffssl_tvc_runs/diffssl_tvc_20260624_153343_diffssl_lstm32_tvc_multisetting/checkpoints/best-096-1181460.ckpt


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         esr/test          │   0.0044409045949578285   │
│         loss/test         │    0.06627386063337326    │
│       loss/test_fd        │    0.1321597397327423     │
│       loss/test_td        │   0.0003879625292029232   │
│         mae/test          │   0.0003879625292029232   │
│         mse/test          │   3.616245578541566e-07   │
│         rmse/test         │  0.00038750070962123573   │
└───────────────────────────┴───────────────────────────┘

[{'loss/test': 0.06627386063337326,
  'loss/test_td': 0.0003879625292029232,
  'loss/test_fd': 0.1321597397327423,
  'mae/test': 0.0003879625292029232,
  'mse/test': 3.616245578541566e-07,
  'esr/test': 0.0044409045949578285,
  'rmse/test': 0.00038750070962123573}]